## Passo 18: Exportação final dos dados do painel
Gera `series.json` e `mapa.geojson` a partir de `df_final_completo` e `gdf_mapa_limpo`.
Rode isso depois que essas duas variáveis já estiverem prontas no seu notebook principal.

In [ ]:
"""
Passo 18: EXPORTACAO FINAL - gera series.json e mapa.geojson
para o painel (index.html), no formato que o site espera.
Rode isso no Colab depois de ter df_final_completo e gdf_mapa_limpo prontos.
"""

import json
import pandas as pd

# ---------- 1. indicadores de composicao (% de participacao no estado, por ano) ----------

df_export = df_final_completo.copy()

# unifica produtividade primaria (nomes de coluna mudam por era)
df_export["PRODUTIVIDADE_UNIFICADA"] = (
    df_export.get("PRODUTIVIDADE_PRIMARIA_ANO2")
    .fillna(df_export.get("MEDIA_PRODUTIVIDADE_PRIMARIA"))
    .fillna(df_export.get("PRODUTIVIDADE_PRIMARIA"))
)

# inverso do VAF per capita (so existe a partir de 2010)
df_export["INVERSO_VAF_PERCAPITA"] = 1 / df_export["VAF_PERCAPITA"]

def calcula_pct_estado(df, coluna):
    return df[coluna] / df.groupby("ANO")[coluna].transform("sum") * 100

df_export["vaf_pct"]            = calcula_pct_estado(df_export, "VAF_ANO_M1")
df_export["area_pct"]           = calcula_pct_estado(df_export, "AREA_KM2")
df_export["pop_pct"]            = calcula_pct_estado(df_export, "POPULACAO")
df_export["prop_rural_pct"]     = calcula_pct_estado(df_export, "PROPRIEDADES_RURAIS")
df_export["pit_pct"]            = calcula_pct_estado(df_export, "PIT")
df_export["pre_educ_pct"]       = calcula_pct_estado(df_export, "PRE_EDUCACAO")
df_export["produtividade_pct"]  = calcula_pct_estado(df_export, "PRODUTIVIDADE_UNIFICADA")
df_export["inverso_vaf_pc_pct"] = calcula_pct_estado(df_export, "INVERSO_VAF_PERCAPITA")

# ---------- 2. series.json (dados aninhados por municipio) ----------

def safe(v):
    return None if pd.isna(v) else round(float(v), 6)

series = {}
for _, row in df_export.iterrows():
    cd = str(int(row["CD_MUN_IBGE"]))
    if cd not in series:
        series[cd] = {
            "cod_sefaz": str(row["COD_SEFAZ"]),
            "nome": row["MUNICIPIO_IBGE"],
            "anos": {}
        }
    series[cd]["anos"][str(int(row["ANO"]))] = {
        "ipm": safe(row["IPM"]),
        "ipm_oficial": bool(row["IPM_OFICIAL"]),
        "var_abs": safe(row["VAR_ABS"]),
        "var_pct": safe(row["VAR_PCT"]),
        "ranking": None if pd.isna(row["RANKING"]) else int(row["RANKING"]),
        "vaf_pct": safe(row["vaf_pct"]),
        "area_pct": safe(row["area_pct"]),
        "pop_pct": safe(row["pop_pct"]),
        "prop_rural_pct": safe(row["prop_rural_pct"]),
        "produtividade_pct": safe(row["produtividade_pct"]),
        "inverso_vaf_pc_pct": safe(row["inverso_vaf_pc_pct"]),
        "pit_pct": safe(row["pit_pct"]),
        "pre_educ_pct": safe(row["pre_educ_pct"]),
    }

with open("/content/series.json", "w", encoding="utf-8") as f:
    json.dump(series, f, ensure_ascii=False, separators=(",", ":"))

print(f"series.json: {len(series)} municipios exportados")

# ---------- 3. mapa.geojson (geometria simplificada + codigo IBGE) ----------

gdf_export = gdf_mapa_limpo.copy()
gdf_export["CD_MUN"] = gdf_export["CD_MUN"].astype(int)

# simplifica a geometria para deixar o arquivo mais leve no site (mantem o formato reconhecivel)
gdf_export["geometry"] = gdf_export["geometry"].simplify(0.004, preserve_topology=True)

gdf_export = gdf_export[["CD_MUN", "geometry"]]
gdf_export.to_file("/content/mapa.geojson", driver="GeoJSON")

print(f"mapa.geojson: {len(gdf_export)} municipios exportados")
print("\nArquivos prontos em /content/series.json e /content/mapa.geojson")
print("Baixe os dois e coloque na MESMA PASTA do index.html no seu repositorio GitHub.")
